In [ ]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq
)
from datasets import Dataset, concatenate_datasets
import evaluate
import numpy as np
import os
import wandb
import subprocess
import tempfile
import json
from sacremoses import MosesDetokenizer

In [ ]:
class TrainingConfig:
    model_name = "facebook/nllb-200-distilled-600M"
    task = "bodo_english_bidirectional_translation"
    learning_rate = 3e-5  
    dropout = 0.3  
    batch_size = 32
    gradient_accumulation_steps = 2
    num_epochs = 2
    max_length = 128
    eval_steps = 1000
    save_steps = 1000
    logging_steps = 500
    generation_max_length = 128
    generation_num_beams = 4
    fp16 = True
    brx_lang = "hin_Deva"
    eng_lang = "eng_Latn"
    metric_for_best_model = "eval_loss"
    greater_is_better = False
    warmup_steps = 500
    weight_decay = 0.01
    comet_model = "Unbabel/wmt22-comet-da"
    comet_batch_size = 32
    comet_use_cpu = False

config = TrainingConfig()

In [ ]:
def load_data_files(brx_file, eng_file):
    with open(brx_file, 'r', encoding='utf-8') as f:
        brx_texts = [line.strip() for line in f.readlines()]
    with open(eng_file, 'r', encoding='utf-8') as f:
        eng_texts = [line.strip() for line in f.readlines()]
    assert len(brx_texts) == len(eng_texts), "Files must have same number of lines"
    return brx_texts, eng_texts

In [ ]:
def create_datasets(train_brx, train_eng, val_brx, val_eng, test_brx, test_eng):
    train_brx_data, train_eng_data = load_data_files(train_brx, train_eng)
    val_brx_data,   val_eng_data   = load_data_files(val_brx,   val_eng)
    test_brx_data,  test_eng_data  = load_data_files(test_brx,  test_eng)


    train_forward = Dataset.from_dict({
        "source":   train_brx_data,
        "target":   train_eng_data,
        "src_lang": [config.brx_lang] * len(train_brx_data),
        "tgt_lang": [config.eng_lang] * len(train_brx_data)
    })

    train_backward = Dataset.from_dict({
        "source":   train_eng_data,
        "target":   train_brx_data,
        "src_lang": [config.eng_lang] * len(train_eng_data),
        "tgt_lang": [config.brx_lang] * len(train_eng_data)
    })
    train_combined = concatenate_datasets([train_forward, train_backward])

    val_forward = Dataset.from_dict({
        "source":   val_brx_data,
        "target":   val_eng_data,
        "src_lang": [config.brx_lang] * len(val_brx_data),
        "tgt_lang": [config.eng_lang] * len(val_brx_data)
    })

    val_backward = Dataset.from_dict({
        "source":   val_eng_data,
        "target":   val_brx_data,
        "src_lang": [config.eng_lang] * len(val_eng_data),
        "tgt_lang": [config.brx_lang] * len(val_eng_data)
    })

    val_combined = concatenate_datasets([val_forward, val_backward])

    test_brx2eng = Dataset.from_dict({
        "source":   test_brx_data,
        "target":   test_eng_data,
        "src_lang": [config.brx_lang] * len(test_brx_data),
        "tgt_lang": [config.eng_lang] * len(test_brx_data)
    })

    test_eng2brx = Dataset.from_dict({
        "source":   test_eng_data,
        "target":   test_brx_data,
        "src_lang": [config.eng_lang] * len(test_eng_data),
        "tgt_lang": [config.brx_lang] * len(test_eng_data)
    })

    return {
        "train":       train_combined,
        "val":         val_combined,
        "val_brx2eng": val_forward,
        "val_eng2brx": val_backward,
        "test_brx2eng": test_brx2eng,
        "test_eng2brx": test_eng2brx,
        "dataset_info": {
            "train_size":          len(train_brx_data),
            "combined_train_size": len(train_brx_data) * 2,
            "val_size":            len(val_brx_data),
            "combined_val_size":   len(val_brx_data) * 2,
            "test_size":           len(test_brx_data),
            "total_size":          len(train_brx_data) * 2 + len(val_brx_data) * 2 + len(test_brx_data)
        }
    }

In [ ]:
def create_preprocess_function(tokenizer):
    def preprocess_data(examples):
        sources = examples["source"]
        targets = examples["target"]
        src_langs = examples["src_lang"]
        tgt_langs = examples["tgt_lang"]

        input_ids = []
        attention_masks = []
        labels = []

        for source, target, src_lang, tgt_lang in zip(sources, targets, src_langs, tgt_langs):
            tokenizer.src_lang = src_lang
            source_encoding = tokenizer(source, max_length=128, truncation=True, padding=False)

            target_tokens = tokenizer(
                target, max_length=126, truncation=True,
                padding=False, add_special_tokens=False
            )["input_ids"]

            tgt_lang_id = tokenizer.convert_tokens_to_ids(tgt_lang)
            target_ids = [tgt_lang_id] + target_tokens + [tokenizer.eos_token_id]

            input_ids.append(source_encoding["input_ids"])
            attention_masks.append(source_encoding["attention_mask"])
            labels.append(target_ids)

        return {
            "input_ids": input_ids,
            "attention_mask": attention_masks,
            "labels": labels
        }
    return preprocess_data


In [ ]:
def compute_comet_score(sources, predictions, references, model_name=None, batch_size=32, use_cpu=False):

    if model_name is None:
        model_name = config.comet_model

    try:
        # Create temporary files for sources, predictions, and references
        with tempfile.NamedTemporaryFile(mode='w', encoding='utf-8', suffix='.txt', delete=False) as src_f, \
             tempfile.NamedTemporaryFile(mode='w', encoding='utf-8', suffix='.txt', delete=False) as hyp_f, \
             tempfile.NamedTemporaryFile(mode='w', encoding='utf-8', suffix='.txt', delete=False) as ref_f:

            # Write data to temporary files
            src_f.write('\n'.join(sources))
            hyp_f.write('\n'.join(predictions))
            ref_f.write('\n'.join(references))

            src_file = src_f.name
            hyp_file = hyp_f.name
            ref_file = ref_f.name

        # Run comet-score command
        cmd = [
            'comet-score',
            '-s', src_file,
            '-t', hyp_file,
            '-r', ref_file,
            '--model', model_name,
            '--batch_size', str(batch_size),
            '--quiet',
            '--only_system'
        ]

        # Force CPU usage if specified or if we detect memory issues
        if use_cpu:
            cmd.extend(['--gpus', '0'])

        result = subprocess.run(
            cmd,
            capture_output=True,
            text=True,
            check=True
        )

        # Parse the output to get the score
        output = result.stdout.strip()

        # Try to parse as JSON first (newer versions output JSON)
        try:
            score_data = json.loads(output)
            if isinstance(score_data, dict) and 'system_score' in score_data:
                score = float(score_data['system_score'])
            else:
                score = float(output)
        except (json.JSONDecodeError, ValueError):
            # Try to extract score from text output
            try:
                score = float(output)
            except ValueError:
                # Look for score in output lines
                for line in output.split('\n'):
                    if 'score' in line.lower():
                        try:
                            score = float(line.split(':')[-1].strip())
                            break
                        except:
                            continue
                else:
                    score = 0.0

        # Clean up temporary files
        os.unlink(src_file)
        os.unlink(hyp_file)
        os.unlink(ref_file)


        return score

    except subprocess.CalledProcessError as e:
        print(f"Warning: comet-score command failed: {e.stderr}")
        # Clean up files if they exist
        try:
            os.unlink(src_file)
            os.unlink(hyp_file)
            os.unlink(ref_file)
        except:
            pass
        return 0.0
    except Exception as e:
        print(f"Warning: COMET computation failed: {str(e)}")
        return 0.0


In [ ]:
def create_compute_metrics_function(tokenizer):
    def compute_metrics(eval_preds):
        predictions, labels = eval_preds

        decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
        labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
        decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

        decoded_preds = [pred.strip() for pred in decoded_preds]
        decoded_labels = [label.strip() for label in decoded_labels]


        # Load basic metrics
        sacrebleu = evaluate.load("sacrebleu")
        chrf = evaluate.load("chrf")
        meteor = evaluate.load("meteor")

        # Compute basic metrics
        sacrebleu_result = sacrebleu.compute(predictions=decoded_preds, references=decoded_labels)
        chrf_result = chrf.compute(predictions=decoded_preds, references=decoded_labels, word_order=0)
        chrfpp_result = chrf.compute(predictions=decoded_preds, references=decoded_labels, word_order=2)
        meteor_result = meteor.compute(predictions=decoded_preds, references=decoded_labels)

        # Compute COMET score using CLI tool
        sources = [""] * len(decoded_preds)
        comet_score = compute_comet_score(
            sources, decoded_preds, decoded_labels,
            batch_size=config.comet_batch_size,
            use_cpu=config.comet_use_cpu
        )

        return {
            "sacrebleu": sacrebleu_result["score"],
            "chrf": chrf_result["score"],
            "chrfpp": chrfpp_result["score"],
            "meteor": meteor_result["meteor"],
            "comet": comet_score
        }
    return compute_metrics

In [ ]:
class SpecializationTrainer(Seq2SeqTrainer):

    def __init__(self, *args, **kwargs):
        self.specialization_lr = kwargs.pop('specialization_lr', 1e-4)
        self.specialization_dropout = kwargs.pop('specialization_dropout', 0.1)
        super().__init__(*args, **kwargs)
        self.current_epoch = 0
        self.samples_per_epoch = None

    def on_train_begin(self, args, state, control, **kwargs):
        dataset_size = len(self.train_dataset)
        batch_size = args.per_device_train_batch_size * args.gradient_accumulation_steps
        if args.world_size > 1:
            batch_size *= args.world_size
        self.samples_per_epoch = dataset_size
        self.steps_per_epoch = dataset_size // batch_size

        # Set initial dropout
        self._update_model_dropout(self.specialization_dropout)

        super().on_train_begin(args, state, control, **kwargs)

    def on_epoch_begin(self, args, state, control, **kwargs):
        if hasattr(state, 'epoch'):
            current_epoch_number = int(state.epoch) + 1
        else:
            current_epoch_number = (state.global_step // self.steps_per_epoch) + 1

        self.current_epoch = current_epoch_number

        # Ensure learning rate and dropout are correct
        for param_group in self.optimizer.param_groups:
            param_group['lr'] = self.specialization_lr

        self._update_model_dropout(self.specialization_dropout)

        wandb.log({
            f"config/epoch_{current_epoch_number}_lr": self.specialization_lr,
            f"config/epoch_{current_epoch_number}_dropout": self.specialization_dropout,
            "epoch": current_epoch_number,
            "phase": "SPECIALIZATION"
        })

    def on_epoch_end(self, args, state, control, **kwargs):
        completed_epoch = int(state.epoch) + 1

    def _update_model_dropout(self, dropout_rate):
        for module in self.model.modules():
            if isinstance(module, torch.nn.Dropout):
                module.p = dropout_rate


In [ ]:
def create_specialization_trainer(model, tokenizer, train_dataset, val_dataset, compute_metrics):
    training_args = Seq2SeqTrainingArguments(
        output_dir="/checkpoint/nllb-200-as-ne",
        num_train_epochs=config.num_epochs,
        per_device_train_batch_size=config.batch_size,
        per_device_eval_batch_size=config.batch_size,
        gradient_accumulation_steps=config.gradient_accumulation_steps,
        learning_rate=config.learning_rate,
        weight_decay=config.weight_decay,
        warmup_steps=config.warmup_steps,
        eval_strategy="steps",
        eval_steps=config.eval_steps,
        save_steps=config.save_steps,
        logging_steps=config.logging_steps,
        predict_with_generate=True,
        generation_max_length=config.generation_max_length,
        generation_num_beams=config.generation_num_beams,
        load_best_model_at_end=True,
        metric_for_best_model=config.metric_for_best_model,
        greater_is_better=True,
        remove_unused_columns=False,
        fp16=config.fp16,
        report_to="wandb",
        run_name="specialization-only-translation",
        logging_dir='./logs',
        save_total_limit=5,
    )

    data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model, padding=True)

    trainer = SpecializationTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        specialization_lr=config.learning_rate,
        specialization_dropout=config.dropout,
    )

    return trainer

In [ ]:
def run_specialization_training(train_brx_file, train_eng_file,
                                 val_brx_file, val_eng_file,
                                 test_brx_file, test_eng_file):

    os.environ["WANDB_API_KEY"] = "enter-wandb-api-key"
    wandb.login()

    print("\n" + "="*80)
    print("STARTING TRAINING PIPELINE (Bodo ↔ English)")
    print("="*80 + "\n")

    wandb_config = {
        "model_name": config.model_name,
        "task": config.task,
        "learning_rate": config.learning_rate,
        "dropout": config.dropout,
        "batch_size": config.batch_size,
        "gradient_accumulation_steps": config.gradient_accumulation_steps,
        "num_epochs": config.num_epochs,
        "max_length": config.max_length,
        "eval_steps": config.eval_steps,
        "save_steps": config.save_steps,
        "logging_steps": config.logging_steps,
        "generation_max_length": config.generation_max_length,
        "generation_num_beams": config.generation_num_beams,
        "fp16": config.fp16,
        "brx_lang": config.brx_lang,
        "eng_lang": config.eng_lang,
        "metric_for_best_model": config.metric_for_best_model,
        "warmup_steps": config.warmup_steps,
        "weight_decay": config.weight_decay,
        "comet_model": config.comet_model,
    }

    run = wandb.init(
        project="Bodo-english-translation",
        name="nllb-as-en",
        config=wandb_config,
        tags=["nepali", "english", "translation", "nllb"]
    )

    print("Loading model and tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(config.model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(config.model_name)

    preprocess_data = create_preprocess_function(tokenizer)
    compute_metrics = create_compute_metrics_function(tokenizer)

    print("Creating datasets...")
    datasets = create_datasets(train_brx_file, train_eng_file,
                              val_brx_file, val_eng_file,
                              test_brx_file, test_eng_file)

    wandb.log(datasets["dataset_info"])

    print("Preprocessing training data...")
    train_dataset = datasets["train"].map(
        preprocess_data, batched=True,
        remove_columns=datasets["train"].column_names
    )

    print("Preprocessing validation data...")
    val_dataset = datasets["val"].map(
        preprocess_data, batched=True,
        remove_columns=datasets["val"].column_names
    )

    trainer = create_specialization_trainer(model, tokenizer, train_dataset, val_dataset, compute_metrics)

    print("\nStarting training...\n")
    trainer.train()


    return trainer, results


In [ ]:
if __name__ == "__main__":
    trainer, results = run_specialization_training(
        train_brx_file="train-data-brx",
        train_eng_file="train-data-en",
        val_brx_file="eval-data-brx",
        val_eng_file="eval-data-en",
        test_brx_file="test-data-brx",
        test_eng_file="test-data-en",
    )


In [ ]:
CHECKPOINT_PATH = "/best/checkpoint"

TEST_BRX_FILE   = "test-data-brx"
TEST_ENG_FILE   = "test-data-en"


COMET_USE_CPU   = False
COMET_MODEL     = "Unbabel/wmt22-comet-da"
COMET_BATCH     = 32

GENERATION_MAX_LENGTH = 128
GENERATION_NUM_BEAMS  = 5
BATCH_SIZE            = 32
MAX_LENGTH            = 128

BRX_LANG = "hin_Deva"
ENG_LANG = "eng_Latn"

In [ ]:
import os, json, subprocess, tempfile
import torch
import numpy as np
import evaluate
from torch.utils.data import DataLoader, Dataset as TorchDataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from sacremoses import MosesDetokenizer

In [ ]:
def load_parallel_files(file_a, file_b):
    with open(file_a, encoding="utf-8") as f:
        lines_a = [l.strip() for l in f]
    with open(file_b, encoding="utf-8") as f:
        lines_b = [l.strip() for l in f]
    assert len(lines_a) == len(lines_b), "Test files must have the same number of lines."
    return lines_a, lines_b

In [ ]:
class TranslationDataset(TorchDataset):

    def __init__(self, src_texts, tgt_texts, tokenizer, src_lang, tgt_lang):
        self.src_texts  = src_texts
        self.tgt_texts  = tgt_texts
        self.tokenizer  = tokenizer
        self.src_lang   = src_lang
        self.tgt_lang   = tgt_lang

    def __len__(self):
        return len(self.src_texts)

    def __getitem__(self, idx):
        self.tokenizer.src_lang = self.src_lang
        enc = self.tokenizer(
            self.src_texts[idx],
            max_length=MAX_LENGTH,
            truncation=True,
            padding=False,
            return_tensors="pt",
        )
        return {
            "input_ids":      enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
        }


def collate_fn(tokenizer):
    pad_id = tokenizer.pad_token_id

    def _collate(batch):
        max_len = max(b["input_ids"].size(0) for b in batch)
        input_ids      = torch.full((len(batch), max_len), pad_id, dtype=torch.long)
        attention_mask = torch.zeros(len(batch), max_len, dtype=torch.long)
        for i, b in enumerate(batch):
            l = b["input_ids"].size(0)
            input_ids[i, :l]      = b["input_ids"]
            attention_mask[i, :l] = b["attention_mask"]
        return {"input_ids": input_ids, "attention_mask": attention_mask}

    return _collate


In [ ]:
def generate_translations(model, tokenizer, src_texts, tgt_texts, src_lang, tgt_lang, device):
    dataset    = TranslationDataset(src_texts, tgt_texts, tokenizer, src_lang, tgt_lang)
    loader     = DataLoader(dataset, batch_size=BATCH_SIZE, collate_fn=collate_fn(tokenizer))
    tgt_lang_id = tokenizer.convert_tokens_to_ids(tgt_lang)

    all_preds = []
    model.eval()
    with torch.no_grad():
        for batch in loader:
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)

            outputs = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                forced_bos_token_id=tgt_lang_id,
                max_length=GENERATION_MAX_LENGTH,
                num_beams=GENERATION_NUM_BEAMS,
            )
            decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
            all_preds.extend(decoded)

    return all_preds


In [ ]:
def compute_all_metrics(predictions, references, sources):
    md = MosesDetokenizer(lang="en")
    predictions = [md.detokenize(p.strip().split()) for p in predictions]
    references  = [md.detokenize(r.strip().split()) for r in references]

    sacrebleu_m = evaluate.load("sacrebleu")
    chrf_m      = evaluate.load("chrf")
    meteor_m    = evaluate.load("meteor")

    sb  = sacrebleu_m.compute(predictions=predictions, references=references)
    cf  = chrf_m.compute(predictions=predictions, references=references, word_order=0)
    cfp = chrf_m.compute(predictions=predictions, references=references, word_order=2)
    mt  = meteor_m.compute(predictions=predictions, references=references)
    comet = compute_comet(sources, predictions, references)

    return {
        "sacrebleu": sb["score"],
        "chrf":      cf["score"],
        "chrfpp":    cfp["score"],
        "meteor":    mt["meteor"],
        "comet":     comet,
    }

In [ ]:
def compute_comet(sources, predictions, references):
    try:
        with (
            tempfile.NamedTemporaryFile(mode="w", encoding="utf-8", suffix=".txt", delete=False) as sf,
            tempfile.NamedTemporaryFile(mode="w", encoding="utf-8", suffix=".txt", delete=False) as hf,
            tempfile.NamedTemporaryFile(mode="w", encoding="utf-8", suffix=".txt", delete=False) as rf,
        ):
            sf.write("\n".join(sources));       src_path = sf.name
            hf.write("\n".join(predictions));   hyp_path = hf.name
            rf.write("\n".join(references));    ref_path = rf.name

        cmd = [
            "comet-score",
            "-s", src_path, "-t", hyp_path, "-r", ref_path,
            "--model", COMET_MODEL,
            "--batch_size", str(COMET_BATCH),
            "--quiet", "--only_system",
        ]
        if COMET_USE_CPU:
            cmd += ["--gpus", "0"]

        result = subprocess.run(cmd, capture_output=True, text=True, check=True)
        output = result.stdout.strip()

        try:
            data  = json.loads(output)
            score = float(data["system_score"]) if "system_score" in data else float(output)
        except (json.JSONDecodeError, ValueError):
            score = 0.0
            for line in output.split("\n"):
                if "score" in line.lower():
                    try:
                        score = float(line.split(":")[-1].strip()); break
                    except Exception:
                        pass

        for p in (src_path, hyp_path, ref_path):
            try: os.unlink(p)
            except Exception: pass

        return score

    except subprocess.CalledProcessError as e:
        print(f"comet-score failed: {e.stderr}")
        return 0.0
    except Exception as e:
        print(f"COMET error: {e}")
        return 0.0

In [ ]:
def main():
    print("\n" + "=" * 80)
    print("REPRODUCIBILITY CHECK — NLLB-200 Bodo ↔ English")
    print("=" * 80)
    print(f"  Checkpoint : {CHECKPOINT_PATH}")
    print(f"  BRX test   : {TEST_BRX_FILE}")
    print(f"  ENG test   : {TEST_ENG_FILE}")
    print("=" * 80 + "\n")

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"  Device: {device}\n")

    print("Loading tokenizer and model from checkpoint...")
    tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT_PATH)
    model     = AutoModelForSeq2SeqLM.from_pretrained(CHECKPOINT_PATH).to(device)

    brx_texts, eng_texts = load_parallel_files(TEST_BRX_FILE, TEST_ENG_FILE)
    print(f"  Test pairs loaded: {len(brx_texts)}\n")

    # ── Bodo → English ──
    print("Generating translations: Bodo → English ...")
    brx2eng_preds = generate_translations(
        model, tokenizer, brx_texts, eng_texts, BRX_LANG, ENG_LANG, device
    )
    print("Computing metrics: Bodo → English ...")
    brx2eng = compute_all_metrics(brx2eng_preds, eng_texts, brx_texts)

    # ── English → Bodo ──
    print("\nGenerating translations: English → Bodo ...")
    eng2brx_preds = generate_translations(
        model, tokenizer, eng_texts, brx_texts, ENG_LANG, BRX_LANG, device
    )
    print("Computing metrics: English → Bodo ...")
    eng2brx = compute_all_metrics(eng2brx_preds, brx_texts, eng_texts)

    # ── Averages ──
    avg = {k: (brx2eng[k] + eng2brx[k]) / 2 for k in brx2eng}

    # ── Print results (same format as original) ──
    print("\n" + "=" * 80)
    print("Bodo TO ENGLISH TRANSLATION RESULTS")
    print("=" * 80)
    print("SacreBLEU: %.4f" % brx2eng["sacrebleu"])
    print("ChrF:      %.4f" % brx2eng["chrf"])
    print("ChrF++:    %.4f" % brx2eng["chrfpp"])
    print("METEOR:    %.4f" % brx2eng["meteor"])
    print("COMET:     %.4f" % brx2eng["comet"])

    print("\n" + "=" * 80)
    print("ENGLISH TO Bodo TRANSLATION RESULTS")
    print("=" * 80)
    print("SacreBLEU: %.4f" % eng2brx["sacrebleu"])
    print("ChrF:      %.4f" % eng2brx["chrf"])
    print("ChrF++:    %.4f" % eng2brx["chrfpp"])
    print("METEOR:    %.4f" % eng2brx["meteor"])
    print("COMET:     %.4f" % eng2brx["comet"])

    print("\n" + "=" * 80)
    print("AVERAGE RESULTS ACROSS BOTH DIRECTIONS")
    print("=" * 80)
    print("Average SacreBLEU: %.4f" % avg["sacrebleu"])
    print("Average ChrF:      %.4f" % avg["chrf"])
    print("Average ChrF++:    %.4f" % avg["chrfpp"])
    print("Average METEOR:    %.4f" % avg["meteor"])
    print("Average COMET:     %.4f" % avg["comet"])
    print("=" * 80 + "\n")

In [ ]:
if __name__ == "__main__":
    main()